In [1]:
from pyspark.sql import SparkSession

APP_NAME = 'etl_clientes'

spark = (
    SparkSession.builder
    .appName(APP_NAME)
    .master("spark://spark-master:7077")
    .config("spark.sql.catalogImplementation", "in-memory")
    .getOrCreate()
)

In [2]:
df_raw = spark.read.option("header", True).csv("file:///mnt/notebooks/clientes_sinteticos.csv")

In [3]:
df_raw.printSchema()

root
 |-- cod_cliente: string (nullable = true)
 |-- nm_cliente: string (nullable = true)
 |-- nm_pais_cliente: string (nullable = true)
 |-- nm_cidade_cliente: string (nullable = true)
 |-- nm_rua_cliente: string (nullable = true)
 |-- num_casa_cliente: string (nullable = true)
 |-- telefone_cliente: string (nullable = true)
 |-- dt_nascimento_cliente: string (nullable = true)
 |-- dt_atualizacao: string (nullable = true)
 |-- tp_pessoa: string (nullable = true)
 |-- vl_renda: string (nullable = true)



In [4]:
df_raw.show(5)

+-----------+----------------+---------------+-----------------+--------------+----------------+--------------------+---------------------+--------------+---------+--------+
|cod_cliente|      nm_cliente|nm_pais_cliente|nm_cidade_cliente|nm_rua_cliente|num_casa_cliente|    telefone_cliente|dt_nascimento_cliente|dt_atualizacao|tp_pessoa|vl_renda|
+-----------+----------------+---------------+-----------------+--------------+----------------+--------------------+---------------------+--------------+---------+--------+
|        980|   Krista Rogers|     Guadeloupe|        Kellyview|   Barker Walk|            3309|      (95)39194-2483|           2001-10-29|    2024-04-18|       PJ|57900.19|
|        177|Veronica Russell|  Guinea-Bissau|        Romanside| Tucker Canyon|             268|      (42)71167-9960|           1991-08-03|    2024-10-28|       PJ|78775.44|
|        267|  William Hughes|       Mongolia|       Taylorview|  Daniel Inlet|           72089|      (70)40026-9752|           20

In [5]:
from pyspark.sql.types import (StructField, StructType, StringType, IntegerType, TimestampType)

schema_clientes = StructType([
    StructField("cod_cliente", StringType(), True),
    StructField("nm_cliente", StringType(), True),
    StructField("nm_pais_cliente", StringType(), True),
    StructField("nm_cidade_cliente", StringType(), True),
    StructField("nm_rua_cliente", StringType(), True),
    StructField("num_casa_cliente", StringType(), True),
    StructField("telefone_cliente", StringType(), True),
    StructField("dt_nascimento_cliente", StringType(), True),
    StructField("dt_atualizacao", StringType(), True),
    StructField("tp_pessoa", StringType(), True),
    StructField("vl_renda", StringType(), True)
])

df = spark.read \
.option("header", True) \
.schema(schema_clientes) \
.csv("file:///mnt/notebooks/clientes_sinteticos.csv")

df.show(5, truncate=False)

+-----------+----------------+---------------+-----------------+--------------+----------------+--------------------+---------------------+--------------+---------+--------+
|cod_cliente|nm_cliente      |nm_pais_cliente|nm_cidade_cliente|nm_rua_cliente|num_casa_cliente|telefone_cliente    |dt_nascimento_cliente|dt_atualizacao|tp_pessoa|vl_renda|
+-----------+----------------+---------------+-----------------+--------------+----------------+--------------------+---------------------+--------------+---------+--------+
|980        |Krista Rogers   |Guadeloupe     |Kellyview        |Barker Walk   |3309            |(95)39194-2483      |2001-10-29           |2024-04-18    |PJ       |57900.19|
|177        |Veronica Russell|Guinea-Bissau  |Romanside        |Tucker Canyon |268             |(42)71167-9960      |1991-08-03           |2024-10-28    |PJ       |78775.44|
|267        |William Hughes  |Mongolia       |Taylorview       |Daniel Inlet  |72089           |(70)40026-9752      |2006-12-08   

In [6]:
from pyspark.sql.functions import (col, upper, lit)
from datetime import datetime

processing_date = datetime.now().strftime("%Y%m%d")

df = df \
.withColumn("nm_cliente", upper(col("nm_cliente"))) \
.withColumnRenamed("telefone_cliente", "num_telefone_cliente") \
.withColumn("anomesdia", lit(processing_date))

df.show(5, truncate=False)

+-----------+----------------+---------------+-----------------+--------------+----------------+--------------------+---------------------+--------------+---------+--------+---------+
|cod_cliente|nm_cliente      |nm_pais_cliente|nm_cidade_cliente|nm_rua_cliente|num_casa_cliente|num_telefone_cliente|dt_nascimento_cliente|dt_atualizacao|tp_pessoa|vl_renda|anomesdia|
+-----------+----------------+---------------+-----------------+--------------+----------------+--------------------+---------------------+--------------+---------+--------+---------+
|980        |KRISTA ROGERS   |Guadeloupe     |Kellyview        |Barker Walk   |3309            |(95)39194-2483      |2001-10-29           |2024-04-18    |PJ       |57900.19|20260126 |
|177        |VERONICA RUSSELL|Guinea-Bissau  |Romanside        |Tucker Canyon |268             |(42)71167-9960      |1991-08-03           |2024-10-28    |PJ       |78775.44|20260126 |
|267        |WILLIAM HUGHES  |Mongolia       |Taylorview       |Daniel Inlet  |7

In [7]:
df.write.mode("append").partitionBy("anomesdia").parquet("file:///mnt/notebooks/bronze/tabela_cliente_landing")

In [8]:
df_silver = spark.read.parquet("file:///mnt/notebooks/bronze/tabela_cliente_landing")

df_silver.show(5, truncate=False)

+-----------+----------------+---------------+-----------------+--------------+----------------+--------------------+---------------------+--------------+---------+--------+---------+
|cod_cliente|nm_cliente      |nm_pais_cliente|nm_cidade_cliente|nm_rua_cliente|num_casa_cliente|num_telefone_cliente|dt_nascimento_cliente|dt_atualizacao|tp_pessoa|vl_renda|anomesdia|
+-----------+----------------+---------------+-----------------+--------------+----------------+--------------------+---------------------+--------------+---------+--------+---------+
|980        |KRISTA ROGERS   |Guadeloupe     |Kellyview        |Barker Walk   |3309            |(95)39194-2483      |2001-10-29           |2024-04-18    |PJ       |57900.19|20260126 |
|177        |VERONICA RUSSELL|Guinea-Bissau  |Romanside        |Tucker Canyon |268             |(42)71167-9960      |1991-08-03           |2024-10-28    |PJ       |78775.44|20260126 |
|267        |WILLIAM HUGHES  |Mongolia       |Taylorview       |Daniel Inlet  |7

In [9]:
df_silver.count()

500

In [10]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, when

window_spec = (
    Window
    .partitionBy("cod_cliente")
    .orderBy(col("dt_atualizacao").desc())
)

df_dedup = (
    df
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

phone_regex = r"^\(\d{2}\)\d{5}-\d{4}$"

final_silver = (
    df_dedup
    .withColumn(
        "num_telefone_cliente",
        when(
            col("num_telefone_cliente").rlike(phone_regex),
            col("num_telefone_cliente")
        ).otherwise(lit(None))
    )
)

final_silver.count()

397

In [11]:
final_silver.show(truncate=False)

+-----------+-------------------+-----------------------+-----------------+-----------------+----------------+--------------------+---------------------+--------------+---------+--------+---------+
|cod_cliente|nm_cliente         |nm_pais_cliente        |nm_cidade_cliente|nm_rua_cliente   |num_casa_cliente|num_telefone_cliente|dt_nascimento_cliente|dt_atualizacao|tp_pessoa|vl_renda|anomesdia|
+-----------+-------------------+-----------------------+-----------------+-----------------+----------------+--------------------+---------------------+--------------+---------+--------+---------+
|296        |SUZANNE KIM        |New Zealand            |North Brian      |Pierce Junctions |17705           |(91)63036-2566      |1994-10-09           |2025-04-01    |PJ       |54751.12|20260126 |
|467        |JACOB MARTIN       |Burkina Faso           |West Kathrynville|Jo Wall          |78577           |(19)28117-8307      |1985-08-30           |2024-11-22    |PJ       |74286.76|20260126 |
|691      

In [12]:
final_silver.write.mode("append").partitionBy("anomesdia").parquet("file:///mnt/notebooks/silver/tb_cliente")

# Análise

In [13]:
from pyspark.sql.functions import (col, to_date, current_date, months_between, floor, avg)

df_updates = (
    df
    .groupBy("cod_cliente")
    .count()
    .orderBy(col("count").desc())
    .limit(5)
)

final_silver = final_silver.withColumn("data_nascimento", to_date("dt_nascimento_cliente", "yyyy-MM-dd"))

final_silver = final_silver.withColumn(
    "idade",
    floor(months_between(current_date(), col("data_nascimento")) / 12)
)

df_idade_media = final_silver.agg(avg("idade"))

df_idade_media.show()

+-----------------+
|       avg(idade)|
+-----------------+
|51.22921914357683|
+-----------------+



In [3]:
"""
Module for client data analysis using PySpark
"""

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, DateType
from datetime import datetime
import os


INPUT_PATH = "file:///mnt/notebooks/clientes_sinteticos.csv"
APP_NAME = "analise_clientes"


def get_spark_session() -> SparkSession:
    """
    Create and return a SparkSession instance
    :return: SparkSession instance
    """

    return (
        SparkSession.builder
        .appName(APP_NAME)
        .getOrCreate()
    )


def read_csv(spark, path) -> DataFrame:
    """
    Read a CSV file into a DataFrame
    :param spark: SparkSession instance
    :param path: Path to the CSV file
    :return: DataFrame with the CSV data
    """

    return (
        spark.read
        .option("header", True)
        .csv(path)
    )


def top_most_updated_clients(df, qty=5) -> DataFrame:
    """
    Calculate the top N clients with the most updates
    :param df: DataFrame with client data
    :param qty: Number of top clients to return
    :return: DataFrame with top N clients
    """
    
    return (
        df
        .groupBy("cod_cliente")
        .agg(F.count("*").alias("update_qty"))
        .orderBy(F.col("update_qty").desc())
        .limit(qty)
    )


def remove_duplicates(df) -> DataFrame:
    """
    Remove duplicate records based on 'cod_cliente', keeping the latest update
    :param df: DataFrame with client data
    :return: DataFrame without duplicates
    """

    df = df.withColumn(
        "dt_atualizacao",
        F.to_timestamp("dt_atualizacao")
    )

    window_spec = (
        Window
        .partitionBy("cod_cliente")
        .orderBy(F.col("dt_atualizacao").desc())
    )

    df_dedup = (
        df
        .withColumn("row_num", F.row_number().over(window_spec))
        .filter(F.col("row_num") == 1)
        .drop("row_num")
    )

    return df_dedup

    
def add_idade(df) -> DataFrame:
    """
    Calculate and add the age of clients based on their birth date
    :param df: DataFrame with client data
    :return: DataFrame with an additional 'idade' column
    """
    return (
        df
        .withColumn(
            "dt_nascimento_cliente",
            F.to_date("dt_nascimento_cliente", "yyyy-MM-dd")
        )
        .withColumn(
            "idade",
            F.floor(F.months_between(F.current_date(), F.col("dt_nascimento_cliente")) / 12).cast("int")
        )
    )



def calculate_avg_age(df) -> DataFrame:
    """
    Calculate the average age of clients
    :param df: DataFrame with client data including 'idade' column
    :return: DataFrame with the average age of clients
    """
    return (
        df
        .filter(F.col("dt_nascimento_cliente").isNotNull())
        .agg(F.round(F.avg("idade"), 2).alias("client_avg_age"))
    )


def main():
    spark = get_spark_session()

    print("Reading dataset")
    df = read_csv(spark, INPUT_PATH)

    print("Top 5 clients with most updates:")
    top_updated_clients = top_most_updated_clients(df, qty=5)
    top_updated_clients.show(truncate=False)

    print("Removing duplicates")
    df_dedup = remove_duplicates(df)

    print("Calculating average age of clients:")
    df_age = add_idade(df_dedup)

    # show 5 biggest ages
    df_age.select("cod_cliente", "dt_nascimento_cliente", "idade") \
        .orderBy(F.col("idade").desc()).show(5, truncate=False)

    #show 5 lowest ages
    df_age.select("cod_cliente", "dt_nascimento_cliente", "idade") \
        .orderBy(F.col("idade").asc()).show(5, truncate=False)

    print("Average age of clients:")
    df_avg_age = calculate_avg_age(df_age)
    df_avg_age.show()

    spark.stop()


main()


Reading dataset
Top 5 clients with most updates:
+-----------+----------+
|cod_cliente|update_qty|
+-----------+----------+
|878        |5         |
|479        |5         |
|396        |5         |
|855        |4         |
|980        |3         |
+-----------+----------+

Removing duplicates
Calculating average age of clients:
+-----------+---------------------+-----+
|cod_cliente|dt_nascimento_cliente|idade|
+-----------+---------------------+-----+
|896        |1944-06-24           |81   |
|127        |1944-10-15           |81   |
|734        |1945-01-06           |81   |
|703        |1945-09-12           |80   |
|938        |1945-04-12           |80   |
+-----------+---------------------+-----+
only showing top 5 rows

+-----------+---------------------+-----+
|cod_cliente|dt_nascimento_cliente|idade|
+-----------+---------------------+-----+
|423        |2007-03-01           |18   |
|570        |2007-02-14           |18   |
|267        |2006-12-08           |19   |
|480        |2